# BERT fake-vs-real classifier on HF-vs-HR (PolitiFact++ & GossipCop++)

Faithful port of `fake-vs-real-news-detection-bert-acc-100.ipynb` (TF/Keras
`bert-base-uncased`), pointed at the LIFE **human-written** subset. **Task = fake-vs-real among
human news:** HF (human_fake) = **0 (fake)**, HR (human_true) = **1 (real)** — the *same cut* as
`run_life_lstm.py` (§7n), so this is a direct **BERT-vs-LSTM head-to-head**.

- Same pipeline: clean (lowercase/stopwords/punct) → BERT tokenize → fine-tune 5 epochs,
  Adam 1e-5, stratified 90/10→90/10 split, Fake=0/Real=1.
- **The source notebook's "100%" is a source-leakage artifact** (ISOT real news is all
  Reuters-formatted). Expect **honest, lower** numbers on LIFE data.
- **Caveats:** PolitiFact++ HF/HR is only **291 articles** (~29 test) → noisy; BERT may still
  beat the LSTM's collapse there. GossipCop++ (**12,252**) is the longer run (~10–15 min on GPU,
  all articles by default). Both are **1:2 fake:real** → watch **fake(HF=0) recall** + the
  confusion matrix, not just accuracy.
- **GPU required** (Runtime → Change runtime type → GPU).

In [ ]:
!pip install -q transformers  # TensorFlow is preinstalled on Colab

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - set Runtime to GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path (and its `import run_life_lstm`) resolve
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

## Run the classifier

Each cell prints the class balance, per-epoch train/val accuracy, then a test
`classification_report` + confusion matrix. PolitiFact++ is quick; GossipCop++ (all ~12k
articles, 5 epochs) takes ~10–15 min on a GPU.

In [ ]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

In [ ]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

## Notes
- **Task = fake-vs-real among human news** (HF=0 vs HR=1) — the same cut as the LSTM (§7n), for a
  BERT-vs-LSTM comparison. It imports `FILE_LABELS` + `load_dataframe` from `run_life_lstm.py`, so
  **sync that file to Drive too** (both live in `lstm_real_vs_fake_code/`).
- Faithful to the source notebook (bert-base-uncased, Adam 1e-5, 5 epochs, stratified 90/10→90/10)
  with 4 documented robustness fixes: subsample capped at dataset size, explicit
  `BinaryCrossentropy(from_logits=True)`, attention mask fed to the model, and `--max_length 256`.
- GossipCop++ uses **all** articles by default (`--sample_size 0`); pass e.g. `--sample_size 1000`
  for the source notebook's fast subsample behavior. No model checkpoints saved.